# Model Evaluation

This notebook evaluates the trained model on the test set.

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from src.models.architecture import get_model
from src.data.data_loader import get_data_loaders
from src.utils.metrics import calculate_metrics, calculate_class_specific_metrics, print_metrics_report

## Load Trained Model

In [ ]:
# Load model
model = get_model("cnn")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load trained weights
checkpoint = torch.load('../models/saved_models/best_model.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print("Model loaded successfully")

## Load Test Data

In [ ]:
# Get test data loader
_, _, test_loader = get_data_loaders()

print(f"Test batches: {len(test_loader)}")

## Evaluate on Test Set

In [ ]:
all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)
        
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())
        all_probabilities.extend(probabilities[:, 1].cpu().numpy())  # Probability for dog class

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)

print(f"Evaluated {len(all_labels)} test samples")

## Calculate Metrics

In [ ]:
# Calculate metrics
metrics = calculate_metrics(all_labels, all_predictions, all_probabilities)
class_metrics = calculate_class_specific_metrics(all_labels, all_predictions)

# Print detailed report
print_metrics_report(metrics, class_metrics)

## Confusion Matrix

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## Sample Predictions

In [ ]:
# Display some sample predictions
fig, axes = plt.subplots(2, 4, figsize=(15, 8))

# Get a batch of test images
test_iter = iter(test_loader)
images, labels = next(test_iter)

with torch.no_grad():
    images = images.to(device)
    outputs = model(images)
    probabilities = torch.softmax(outputs, dim=1)
    _, predicted = torch.max(outputs.data, 1)

class_names = ['Cat', 'Dog']

for i in range(8):
    ax = axes[i // 4, i % 4]
    
    # Convert image back to display format
    img = images[i].cpu().permute(1, 2, 0).numpy()
    img = (img * 255).astype(np.uint8)
    
    ax.imshow(img)
    actual = class_names[labels[i]]
    pred = class_names[predicted[i]]
    conf = probabilities[i][predicted[i]].item() * 100
    
    color = 'green' if actual == pred else 'red'
    ax.set_title(f'Actual: {actual}\nPred: {pred} ({conf:.1f}%)', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()